# 06 · Source identities and catalog updates

This notebook uses two **fictional** Overpass-format snapshots. It demonstrates source identity, controlled catalog updates, duplicate-review decisions, and an explicit cross-source link. No network access or model credentials are required.

The catalog (what Savoroute knows about an item) and the source registry (which provider records refer to it) are separate datasets.

## 1. Import a source snapshot

The v0.3.0 importer creates catalog items for admitted source records and returns an import report. A duplicate OSM element remains in the review queue.

In [ ]:
import json
from collections import Counter
from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory

from savoroute.domain.models import Catalog
from savoroute.identity.registry import bootstrap_osm_registry, link_source, load_registry
from savoroute.ingestion.osm import import_osm_snapshot
from savoroute.services.osm_sync import sync_osm_snapshot, write_osm_sync
from savoroute.data.catalog import load_catalog

samples = Path("data/sample")
original = json.loads((samples / "osm_overpass_fixture.json").read_text(encoding="utf-8"))
updated = json.loads((samples / "osm_overpass_update_fixture.json").read_text(encoding="utf-8"))
initial = import_osm_snapshot(original, synthetic_fixture=True)
print("admission decisions:", dict(Counter(d.status for d in initial.decisions)))
print("catalog items:", len(initial.catalog.items))

## 2. Establish the source registry

Bindings map `(provider, source_ref)` to an existing `item_id`. Bootstrapping checks the catalog digest stored in the original import report. Reviewed-but-unadmitted records are **not** bound.

In [ ]:
registry = bootstrap_osm_registry(initial.catalog, initial.report())
print("bindings:", len(registry.bindings))
for binding in registry.bindings:
    print(binding.provider, binding.source_ref, "→", binding.item_id)
assert registry.lookup("osm", "way/900000002") is None

## 3. Reconcile a second snapshot

One record has a new cuisine tag, another is a candidate duplicate, and one is new. The remaining previously imported records do not appear in this bounded update snapshot; absence alone is not interpreted as deletion.

In [ ]:
synced = sync_osm_snapshot(updated, base_catalog=initial.catalog,
                           registry=registry, synthetic_fixture=True)
for d in synced.decisions:
    print(f"{d.source_ref:21} {d.status:9} {d.reason}")
print("catalog items:", len(synced.catalog.items))

## 4. Check identity stability and idempotency

The existing item retains its `item_id` after the attribute update. Replaying exactly the same snapshot produces no catalog changes.

In [ ]:
source_ref = "node/900000001"
old_binding = registry.lookup("osm", source_ref)
new_binding = synced.registry.lookup("osm", source_ref)
assert old_binding.item_id == new_binding.item_id
print("stable item ID:", old_binding.item_id)
print("updated cuisines:", synced.catalog.get(old_binding.item_id).cuisines)
replayed = sync_osm_snapshot(updated, base_catalog=synced.catalog,
                             registry=synced.registry, synthetic_fixture=True)
print("replay decisions:", [d.status for d in replayed.decisions])
assert replayed.catalog == synced.catalog
assert replayed.registry == synced.registry

## 5. Record a reviewed cross-source association

The following identifier is invented for the tutorial. `link_source` records a human-approved association; it does not import or blend the other source's attributes into the catalog.

In [ ]:
linked = link_source(synced.registry, synced.catalog,
                     provider="localguide", source_ref="sample-business-17",
                     item_id=old_binding.item_id)
print(linked.lookup("localguide", "sample-business-17"))


## 6. Protect curator-edited records

The source registry retains the digest of the last source-managed catalog item. If a catalog item is edited outside the reconciliation pipeline, the next source update is held for review.

In [ ]:
edited_item = replace(synced.catalog.get(old_binding.item_id),
                      description="Curator-maintained note")
edited_catalog = Catalog(
    1, synced.catalog.dataset_name, synced.catalog.synthetic,
    tuple(edited_item if i.item_id == edited_item.item_id else i for i in synced.catalog.items),
    synced.catalog.media,
)
next_input = {"savoroute_fixture": True, "elements": [updated["elements"][0]]}
held = sync_osm_snapshot(next_input, base_catalog=edited_catalog,
                         registry=synced.registry, synthetic_fixture=True)
print(held.decisions[0].status, held.decisions[0].reason)
assert held.decisions[0].status == "review"

## 7. Serialize outputs

The catalog, registry, and decisions are written as separate JSON documents. Each file replacement is atomic, but the three-file write is **not** a transaction.

In [ ]:
with TemporaryDirectory() as directory:
    folder = Path(directory)
    write_osm_sync(synced, catalog_path=folder / "catalog.json",
                   registry_path=folder / "registry.json",
                   report_path=folder / "report.json")
    saved_catalog = load_catalog(folder / "catalog.json")
    saved_registry = load_registry(folder / "registry.json", saved_catalog)
    saved_report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
    assert saved_catalog == synced.catalog
    assert saved_registry == synced.registry
    print("report schema:", saved_report["osm_sync_schema_version"])
    print("record decisions:", len(saved_report["decisions"]))

## Interpretation

This is deterministic *identity management*, not learned entity resolution. A shared name or nearby coordinates can nominate a review candidate but cannot confirm that two records describe the same physical establishment. The example is synthetic; precision and recall on independently annotated real restaurant pairs have not been measured.

Continue with [entity identity and updates](../docs/entity-identity-and-updates.md) for the CLI workflow and limitations.